Version: 02.14.2023

# Capstone Project: Bringing It All Together

In this lab, you will bring together many of the tools and techniques that you have learned throughout this course into a final project. You can choose from many different paths to get to the solution. You could use AWS Managed Services, such as Amazon Comprehend, or use the Amazon SageMaker models. Have fun on whichever path you choose.

### Business scenario

You work for a training organization that recently developed an introductory course about machine learning (ML). The course includes more than 40 videos that cover a broad range of ML topics. You have been asked to create an application that will students can use to quickly locate and view video content by searching for topics and key phrases.

You have downloaded all of the videos to an Amazon Simple Storage Service (Amazon S3) bucket. Your assignment is to produce a dashboard that meets your supervisor’s requirements.

To assist you, all of the previous labs have been provided in this workspace.

## Lab steps

To complete this lab, you will follow these steps:

1. [Viewing the video files](#1.-Viewing-the-video-files)
2. [Transcribing the videos](#2.-Transcribing-the-videos)
3. [Normalizing the text](#3.-Normalizing-the-text)
4. [Extracting key phrases and topics](#4.-Extracting-key-phrases-and-topics)
5. [Creating the dashboard](#5.-Creating-the-dashboard)

## Submitting your work

1. In the lab console, choose **Submit** to record your progress and when prompted, choose **Yes**.

1. If the results don't display after a couple of minutes, return to the top of these instructions and choose **Grades**.

     **Tip**: You can submit your work multiple times. After you change your work, choose **Submit** again. Your last submission is what will be recorded for this lab.

1. To find detailed feedback on your work, choose **Details** followed by **View Submission Report**.

## Useful information

The following cell contains some information that might be useful as you complete this project.

In [ ]:
bucket = "c217930a5501403l15995208t1w967515893790-labbucket-79bpucuuvkjn"
job_data_access_role = 'arn:aws:iam::967515893790:role/service-role/c217930a5501403l15995208t1-ComprehendDataAccessRole-iShTFFAvcoGA'

In [ ]:
import os
import re
import json
import time
import uuid
import unicodedata
import subprocess

import boto3
from botocore.exceptions import ClientError

REGION = "us-east-1"

session = boto3.Session(region_name=REGION)

sts_client = session.client("sts", region_name=REGION)
s3_client = session.client("s3", region_name=REGION)
transcribe_client = session.client("transcribe", region_name=REGION)
comprehend_client = session.client("comprehend", region_name=REGION)
es_client = session.client("es", region_name=REGION)

identity = sts_client.get_caller_identity()
expected_account = job_data_access_role.split(":")[4]

lab_bucket_region = (
    s3_client.get_bucket_location(Bucket=bucket).get("LocationConstraint")
    or "us-east-1"
)

print("Notebook account:", identity["Account"])
print("Expected account:", expected_account)
print("Current ARN:", identity["Arn"])
print("API region:", REGION)
print("Lab bucket:", bucket)
print("Lab bucket region:", lab_bucket_region)

assert identity["Account"] == expected_account, (
    "Account mismatch! The bucket/role in the cell above belong to a different "
    "lab session than the one you are logged into. Re-copy 'bucket' and "
    "'job_data_access_role' from THIS lab, then re-run."
)
assert lab_bucket_region == REGION

print("\nSETUP CHECK: PASS")

## 1. Viewing the video files
([Go to top](#Capstone-8:-Bringing-It-All-Together))


The source video files are located in the following shared Amazon Simple Storage Service (Amazon S3) bucket.

In [ ]:
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/

In [ ]:
SOURCE_VIDEO_NAME = "Mod04_Intro.mp4"

source_video_uri = (
    "s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/"
    f"{SOURCE_VIDEO_NAME}"
)

local_video_path = f"/tmp/{SOURCE_VIDEO_NAME}"
lab_video_key = f"capstone/input/{SOURCE_VIDEO_NAME}"
media_video_uri = f"s3://{bucket}/{lab_video_key}"

print("Source:", source_video_uri)
print("Destination:", media_video_uri)

# Copy from the shared course bucket down to local, then up into the lab bucket.
subprocess.run(
    ["aws", "s3", "cp", source_video_uri, local_video_path, "--only-show-errors"],
    check=True,
)
subprocess.run(
    ["aws", "s3", "cp", local_video_path, media_video_uri, "--only-show-errors"],
    check=True,
)

video_head = s3_client.head_object(Bucket=bucket, Key=lab_video_key)
print("Copied bytes:", video_head["ContentLength"])
assert video_head["ContentLength"] > 0

print("\nVIDEO COPY: PASS")

## 2. Transcribing the videos
 ([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to implement your solution to transcribe the videos.

We convert the course MP4 to a mono 16 kHz PCM WAV (the same media shape used in the Lab 7.1 Transcribe example) and upload it to the lab bucket, then start an Amazon Transcribe job on it.

In [ ]:
# Amazon SageMaker notebooks may not ship a system ffmpeg. imageio-ffmpeg
# bundles a static ffmpeg binary we can call directly. Safe to re-run.
try:
    import imageio_ffmpeg
except ModuleNotFoundError:
    subprocess.run(
        ["pip", "install", "-q", "imageio-ffmpeg==0.6.0"],
        check=True,
    )
    import imageio_ffmpeg

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()
print("ffmpeg:", FFMPEG)

In [ ]:
# Download the course MP4 locally (if not already present) and convert to
# mono/16kHz/PCM-16 WAV, which is an ideal input for Amazon Transcribe.
local_mp4 = f"/tmp/{SOURCE_VIDEO_NAME}"
local_wav = "/tmp/transcribe_input.wav"

if not os.path.exists(local_mp4):
    subprocess.run(
        ["aws", "s3", "cp", media_video_uri, local_mp4, "--only-show-errors"],
        check=True,
    )

subprocess.run(
    [FFMPEG, "-y", "-i", local_mp4,
     "-ac", "1", "-ar", "16000", "-c:a", "pcm_s16le",
     local_wav],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

wav_key = "lab71/transcribe-sample/test.wav"
wav_media_uri = f"s3://{bucket}/{wav_key}"

s3_client.upload_file(local_wav, bucket, wav_key)

wav_head = s3_client.head_object(Bucket=bucket, Key=wav_key)
print("WAV uploaded:", wav_media_uri)
print("WAV bytes:", wav_head["ContentLength"])
assert wav_head["ContentLength"] > 0

print("\nWAV UPLOAD: PASS")

In [ ]:
transcribe_job_name = f"transcribe-job-{uuid.uuid1()}"
transcribe_output_key = "transcribe_output.json"

transcribe_start_response = transcribe_client.start_transcription_job(
    TranscriptionJobName=transcribe_job_name,
    Media={"MediaFileUri": wav_media_uri},
    MediaFormat="wav",
    LanguageCode="en-US",
    OutputBucketName=bucket,
    OutputKey=transcribe_output_key,
)

print("Transcribe job started")
print("Name:", transcribe_job_name)
print("Input:", wav_media_uri)
print("Output:", f"s3://{bucket}/{transcribe_output_key}")

In [ ]:
while True:
    transcribe_job = transcribe_client.get_transcription_job(
        TranscriptionJobName=transcribe_job_name
    )["TranscriptionJob"]

    transcribe_status = transcribe_job["TranscriptionJobStatus"]
    print(time.strftime("%H:%M:%S"), transcribe_status)

    if transcribe_status in {"COMPLETED", "FAILED"}:
        break
    time.sleep(15)

if transcribe_status == "FAILED":
    raise RuntimeError(
        transcribe_job.get("FailureReason", "Transcribe job failed.")
    )

print("\nAMAZON TRANSCRIBE JOB: COMPLETED")

In [ ]:
transcript_object = s3_client.get_object(
    Bucket=bucket, Key=transcribe_output_key
)

transcript_payload = json.loads(
    transcript_object["Body"].read().decode("utf-8")
)

transcript_text = (
    transcript_payload
    .get("results", {})
    .get("transcripts", [{}])[0]
    .get("transcript", "")
    .strip()
)

if not transcript_text:
    raise RuntimeError("The Transcribe output contains no transcript.")

print("Transcript characters:", len(transcript_text))
print()
print(transcript_text[:1500])

In [ ]:
task1 = transcribe_client.get_transcription_job(
    TranscriptionJobName=transcribe_job_name
)["TranscriptionJob"]

print("Job name:", task1["TranscriptionJobName"])
print("Status:", task1["TranscriptionJobStatus"])
print("Media format:", task1.get("MediaFormat"))
print("Media URI:", task1["Media"]["MediaFileUri"])
print("Created:", task1["CreationTime"])
print("Transcript:", task1["Transcript"]["TranscriptFileUri"])

assert task1["TranscriptionJobName"].startswith("transcribe-job-")
assert task1["TranscriptionJobStatus"] == "COMPLETED"

print("\nTASK 1 RESOURCE CHECK: PASS")

## 3. Normalizing the text
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to perform any text normalization steps that are necessary for your solution.

In [ ]:
def normalize_text(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s'-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


normalized_transcript = normalize_text(transcript_text)

if not normalized_transcript:
    raise RuntimeError("Normalization produced an empty document.")

print("Original characters:", len(transcript_text))
print("Normalized characters:", len(normalized_transcript))
print()
print(normalized_transcript[:1000])

print("\nNORMALIZATION CHECK: PASS")

## 4. Extracting key phrases and topics
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to extract the key phrases and topics from the videos.

We write the normalized transcript to Amazon S3 as plain text, then start an Amazon Comprehend key-phrases detection job that reads it and writes results back to the lab bucket.

In [ ]:
# Use a dedicated input key for Comprehend so it never collides with the
# Transcribe output object.
comprehend_input_key = "comprehend/input/normalized_transcript.txt"
comprehend_output_prefix = "comprehend-output/"

s3_client.put_object(
    Bucket=bucket,
    Key=comprehend_input_key,
    Body=(normalized_transcript + "\n").encode("utf-8"),
    ContentType="text/plain",
)

comprehend_input_uri = f"s3://{bucket}/{comprehend_input_key}"
comprehend_output_uri = f"s3://{bucket}/{comprehend_output_prefix}"

input_head = s3_client.head_object(Bucket=bucket, Key=comprehend_input_key)
print("Input:", comprehend_input_uri)
print("Output:", comprehend_output_uri)
print("Input bytes:", input_head["ContentLength"])
assert input_head["ContentLength"] > 0

print("\nCOMPREHEND INPUT CHECK: PASS")

In [ ]:
kpe_job_name = f"kpe-job-{uuid.uuid1()}"

kpe_start_response = comprehend_client.start_key_phrases_detection_job(
    InputDataConfig={
        "S3Uri": comprehend_input_uri,
        "InputFormat": "ONE_DOC_PER_LINE",
    },
    OutputDataConfig={"S3Uri": comprehend_output_uri},
    DataAccessRoleArn=job_data_access_role,
    JobName=kpe_job_name,
    LanguageCode="en",
)

kpe_job_id = kpe_start_response["JobId"]
print("Key phrase job started")
print("Name:", kpe_job_name)
print("ID:", kpe_job_id)
print("Initial status:", kpe_start_response["JobStatus"])

In [ ]:
while True:
    kpe_job = comprehend_client.describe_key_phrases_detection_job(
        JobId=kpe_job_id
    )["KeyPhrasesDetectionJobProperties"]

    kpe_status = kpe_job["JobStatus"]
    print(time.strftime("%H:%M:%S"), kpe_status)

    if kpe_status in {"COMPLETED", "FAILED", "STOPPED"}:
        break
    time.sleep(20)

if kpe_status != "COMPLETED":
    raise RuntimeError(
        kpe_job.get("Message", f"Job ended with status {kpe_status}")
    )

print("\nKEY PHRASE DETECTION JOB: COMPLETED")

In [ ]:
task2 = comprehend_client.describe_key_phrases_detection_job(
    JobId=kpe_job_id
)["KeyPhrasesDetectionJobProperties"]

print("Job name:", task2["JobName"])
print("Job ID:", task2["JobId"])
print("Status:", task2["JobStatus"])
print("Input:", task2["InputDataConfig"]["S3Uri"])
print("Output:", task2["OutputDataConfig"]["S3Uri"])

assert task2["JobName"].startswith("kpe-job-")
assert task2["JobStatus"] == "COMPLETED"

print("\nTASK 2 RESOURCE CHECK: PASS")

## 5. Creating the dashboard
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to create the dashboard for your solution.

We create an Amazon Elasticsearch (OpenSearch) domain named `nlp-lab`, matching the Lab 5.2 configuration, including an IP-scoped access policy so Kibana/OpenSearch Dashboards is reachable.

In [ ]:
# Detect this instance's public IP to scope the Elasticsearch access policy,
# matching the Lab 5.2 pattern (PUBLIC_IP/24).
import urllib.request

def _get_public_ip():
    for url in (
        "https://checkip.amazonaws.com",
        "http://checkip.amazonaws.com",
    ):
        try:
            return urllib.request.urlopen(url, timeout=10).read().decode().strip()
        except Exception:
            continue
    raise RuntimeError("Could not determine public IP for the access policy.")

public_ip = _get_public_ip()
access_cidr = public_ip + "/24"
print("Public IP:", public_ip)
print("Access CIDR:", access_cidr)

In [ ]:
ELASTICSEARCH_DOMAIN = "nlp-lab"

access_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Sid": "",
            "Effect": "Allow",
            "Principal": {"AWS": "*"},
            "Action": "es:*",
            "Resource": (
                f"arn:aws:es:{REGION}:{expected_account}:"
                f"domain/{ELASTICSEARCH_DOMAIN}/*"
            ),
            "Condition": {"IpAddress": {"aws:SourceIp": access_cidr}},
        }
    ],
})

try:
    domain_status = es_client.describe_elasticsearch_domain(
        DomainName=ELASTICSEARCH_DOMAIN
    )["DomainStatus"]
    print("Domain already exists.")
    print("Processing:", domain_status["Processing"])

except ClientError as exc:
    if exc.response.get("Error", {}).get("Code") != "ResourceNotFoundException":
        raise

    create_domain_response = es_client.create_elasticsearch_domain(
        DomainName=ELASTICSEARCH_DOMAIN,
        ElasticsearchVersion="7.9",
        ElasticsearchClusterConfig={
            "InstanceType": "t3.small.elasticsearch",
            "InstanceCount": 2,
            "DedicatedMasterEnabled": False,
            "ZoneAwarenessEnabled": False,
        },
        EBSOptions={
            "EBSEnabled": True,
            "VolumeType": "gp2",
            "VolumeSize": 10,
        },
        AccessPolicies=access_policy,
    )
    print("Elasticsearch domain creation started.")
    print("Domain:", create_domain_response["DomainStatus"]["DomainName"])

In [ ]:
domain_start_time = time.time()
domain_timeout_seconds = 45 * 60

while True:
    domain_status = es_client.describe_elasticsearch_domain(
        DomainName=ELASTICSEARCH_DOMAIN
    )["DomainStatus"]

    domain_endpoint = domain_status.get("Endpoint") or domain_status.get("Endpoints")

    print(time.strftime("%H:%M:%S"), {
        "Created": domain_status.get("Created"),
        "Deleted": domain_status.get("Deleted"),
        "Processing": domain_status.get("Processing"),
        "EndpointAvailable": bool(domain_endpoint),
    })

    domain_ready = (
        domain_status.get("Created") is True
        and domain_status.get("Deleted") is False
        and domain_status.get("Processing") is False
        and bool(domain_endpoint)
    )
    if domain_ready:
        break

    if time.time() - domain_start_time > domain_timeout_seconds:
        raise TimeoutError("Domain is still provisioning after 45 minutes.")
    time.sleep(30)

print("\nELASTICSEARCH DOMAIN: ACTIVE")
print("Endpoint:", domain_endpoint)

In [ ]:
task3 = es_client.describe_elasticsearch_domain(
    DomainName="nlp-lab"
)["DomainStatus"]

task3_endpoint = task3.get("Endpoint") or task3.get("Endpoints")

print("Domain:", task3["DomainName"])
print("Version:", task3["ElasticsearchVersion"])
print("Created:", task3["Created"])
print("Deleted:", task3["Deleted"])
print("Processing:", task3["Processing"])
print("Endpoint:", task3_endpoint)

assert task3["DomainName"] == "nlp-lab"
assert task3["Created"] is True
assert task3["Deleted"] is False
assert task3["Processing"] is False
assert task3_endpoint

print("\nTASK 3 RESOURCE CHECK: PASS")

## Final resource check

Confirms that all three graded resources exist and are healthy in this account before you Submit.

In [ ]:
final_identity = sts_client.get_caller_identity()

final_task1 = transcribe_client.get_transcription_job(
    TranscriptionJobName=transcribe_job_name
)["TranscriptionJob"]

final_task2 = comprehend_client.describe_key_phrases_detection_job(
    JobId=kpe_job_id
)["KeyPhrasesDetectionJobProperties"]

final_task3 = es_client.describe_elasticsearch_domain(
    DomainName="nlp-lab"
)["DomainStatus"]

final_es_endpoint = final_task3.get("Endpoint") or final_task3.get("Endpoints")

print("AWS account:", final_identity["Account"])
print("Task 1:", final_task1["TranscriptionJobName"], final_task1["TranscriptionJobStatus"])
print("Task 2:", final_task2["JobName"], final_task2["JobStatus"])
print("Task 3:", final_task3["DomainName"], {
    "Created": final_task3["Created"],
    "Deleted": final_task3["Deleted"],
    "Processing": final_task3["Processing"],
    "EndpointAvailable": bool(final_es_endpoint),
})

assert final_identity["Account"] == expected_account
assert final_task1["TranscriptionJobStatus"] == "COMPLETED"
assert final_task2["JobStatus"] == "COMPLETED"
assert final_task3["Created"] is True
assert final_task3["Deleted"] is False
assert final_task3["Processing"] is False
assert final_es_endpoint

print()
print("ALL THREE GRADER RESOURCES ARE READY")

# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

*©2023 Amazon Web Services, Inc. or its affiliates. All rights reserved. This work may not be reproduced or redistributed, in whole or in part, without prior written permission from Amazon Web Services, Inc. Commercial copying, lending, or selling is prohibited. All trademarks are the property of their owners.*